# 05 — Train SD3.5 inpaint-EDIT LoRA (on PIPE pairs)

Learns to fill a masked region so the inserted object matches the photo, from
PIPE (source, target, mask) pairs. Conditioning validated by the overfit spike
(notebook 00, collapse_ratio 0.18).

Produces TWO artifacts (both required at inference):
`adapter/pytorch_lora_weights.safetensors` + `adapter/input_adapter.pt`.

## 1. Setup (TWO cells) — install pins + restart, then verify

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
!pip install -q --force-reinstall --no-deps 'transformers==4.46.3' 'tokenizers==0.20.3' 'huggingface_hub==0.25.2'
!pip install -q 'diffusers==0.31.0' 'accelerate==0.34.2' 'peft==0.13.2' 'datasets>=2.20' 'safetensors>=0.4.3' 'sentencepiece' 'protobuf' 'pillow>=10' numpy
print('Installed. Restarting kernel...'); os._exit(0)

In [ ]:
# Run AFTER restart
import os, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.insert(0, '/kaggle/working/VIN')
import transformers, diffusers, torch
print('transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert transformers.__version__ == '4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(); print('GPU', torch.cuda.get_device_name(0))

## 2. SD3.5 access (gated) — mount or HF_TOKEN

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local); print('local mount', SD35_MODEL)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN (Read token + agreed access) or a local SD3.5 mount'
    from huggingface_hub import login; login(token=HF_TOKEN); print('HF login OK')

## 3. Smoke train (200 steps, 200 samples) — confirm loss trends down

In [ ]:
from LoRA.train.train_inpaint_edit import run_training
WORK = Path('/kaggle/working/vin_lora')
smoke = run_training(WORK, base_model_id=SD35_MODEL, hf_token=HF_TOKEN,
                     max_train_steps=200, num_train_samples=200)
smoke['provenance']

## 4. Full train (uses config: 1000 steps, 4000 samples)
Run after the smoke looks healthy.

In [ ]:
train = run_training(WORK, base_model_id=SD35_MODEL, hf_token=HF_TOKEN)
print(train['run_dir'])
print('LoRA + input_adapter ->', train['adapter_dir'])
train['provenance']

## 5. Zip artifacts

In [ ]:
import shutil
z = shutil.make_archive(str(train['run_dir']), 'zip', str(train['run_dir']))
print('zip ->', z)